In [0]:
#Initialize
import uuid
from pyspark.sql import functions as F
from datetime import datetime

catalog_name = "workspace"
bronze_database_name = "amazon_fullfilment_bronze"
silver_database_name = "amazon_fullfilment_silver"
gold_database_name = "amazon_fullfilment_gold"

bronze_volume_path ="/Volumes/workspace/amazon_fullfilment_bronze/landing_zone/"
silver_volume_path ="/Volumes/workspace/amazon_fullfilment_silver/processing_zone/"
source_volume_path = "/Volumes/workspace/default/amazon_fullfilment/source/"

# 1. Define the widget (Key name must match the key in the Job UI)
# Syntax: dbutils.widgets.text("key_name", "default_value", "Label")
dbutils.widgets.text("batch_id", "dev_default")

# 2. Capture the value into a variable
current_run_uuid = dbutils.widgets.get("batch_id")
print(f"The Batch ID for this run is: {current_run_uuid}")

# add the 3 metadata to the bronze layer tables
def add_bronze_metadata(df):
    """
    Standardizes the metadata for all Bronze tables.
    """
    return df.withColumn("_ingest_timestamp", F.current_timestamp()) \
             .withColumn("_source_file", F.col("_metadata.file_path")) \
             .withColumn("_batch_id", F.lit(current_run_uuid))

# dynamically get the current shift
def get_current_shift():
    # Get the current hour (0-23)
    current_hour = datetime.now().hour
    
    if 0 <= current_hour < 8:
        return 'Overnight'
    elif 8 <= current_hour < 16:
        return 'Morning'
    else:
        return 'Afternoon'

current_shift = get_current_shift()


In [0]:
# Code modularization: Insert into Bronze layer function
 
def ingest_to_bronze(table_name, schema, source_path, checkpoint_path):
    """
    Standardizes the Auto Loader ingestion for any table.
    """
    raw_stream = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .schema(schema)
        .load(source_path))
    
    # We select * and _metadata to ensure the global function works
    return (add_bronze_metadata(raw_stream.select("*", "_metadata"))
        .writeStream
        .option("checkpointLocation", f"{checkpoint_path}/_checkpoint")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"{catalog_name}.{bronze_database_name}.{table_name}"))